In [254]:
import yfinance as yf
import pandas as pd

In [255]:
tickers = {
    "oil": "CL=F",
    "xle": "XLE",
    "xom": "XOM",
    "cvx": "CVX",
    "cop": "COP",
    "bp": "BP"
}

data = {}
for name, symbol in tickers.items():
    df = yf.download(symbol, start="2025-09-01", end="2026-06-30")
    df.to_csv(f"data/{name}.csv")
    data[name] = df
    print(f"{name} ({symbol}): {len(df)} rows pulled")

oil = data["oil"]
xle = data["xle"]
xom = data["xom"]
cvx = data["cvx"]
cop = data["cop"]
bp = data["bp"]

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

oil (CL=F): 207 rows pulled
xle (XLE): 207 rows pulled
xom (XOM): 207 rows pulled
cvx (CVX): 207 rows pulled
cop (COP): 207 rows pulled
bp (BP): 207 rows pulled


In [256]:
events = pd.DataFrame([
    {
        "date": "2026-02-28",
        "event": "US and Israel launch war on Iran",
        "category": "military",
        "direction": "baseline",
        "summary": "War begins on a Saturday; markets closed, real reaction shows up the following Monday."
    },
    {
        "date": "2026-03-06",
        "event": "Export halt fails to resolve; grace period ends",
        "category": "military/escalation",
        "direction": "up",
        "summary": "Brent rose $7.28 in a single day, capping a 28% cumulative rise from pre-war levels as markets realized the disruption would not be brief."
    },
    {
        "date": "2026-04-08",
        "event": "8-day ceasefire announced",
        "category": "diplomatic/de-escalation",
        "direction": "down",
        "summary": "Futures dropped sharply on ceasefire news even though spot prices stayed elevated near $124.68, since physical supply disruption had not yet resolved."
    },
    {
        "date": "2026-05-20",
        "event": "Trump says talks in 'final stages,' strikes called off",
        "category": "diplomatic/de-escalation",
        "direction": "down",
        "summary": "WTI and Brent both fell over 5%, pushing crude below $100/barrel for the first time in weeks."
    },
    {
        "date": "2026-06-16",
        "event": "Peace framework signed at G7; Strait of Hormuz to fully reopen",
        "category": "diplomatic/de-escalation",
        "direction": "down",
        "summary": "Brent fell 5% to $78.96 (first sub-$80 close since March); WTI fell 5.8% to $76.05."
    }
])

events.to_csv("data/events.csv", index=False)
events

,date,event,category,direction,summary
0,2026-02-28,US and Israel launch war on Iran,military,baseline,"War begins on a Saturday; markets closed, real..."
1,2026-03-06,Export halt fails to resolve; grace period ends,military/escalation,up,"Brent rose $7.28 in a single day, capping a 28..."
2,2026-04-08,8-day ceasefire announced,diplomatic/de-escalation,down,Futures dropped sharply on ceasefire news even...
3,2026-05-20,"Trump says talks in 'final stages,' strikes ca...",diplomatic/de-escalation,down,"WTI and Brent both fell over 5%, pushing crude..."
4,2026-06-16,Peace framework signed at G7; Strait of Hormuz...,diplomatic/de-escalation,down,Brent fell 5% to $78.96 (first sub-$80 close s...


In [257]:
stock_names = ["xom", "cvx", "cop", "bp"]

returns = pd.DataFrame({
    "oil": oil["Close"].squeeze().pct_change() * 100,
    "xle": xle["Close"].squeeze().pct_change() * 100,
    "xom": xom["Close"].squeeze().pct_change() * 100,
    "cvx": cvx["Close"].squeeze().pct_change() * 100,
    "cop": cop["Close"].squeeze().pct_change() * 100,
    "bp": bp["Close"].squeeze().pct_change() * 100,
})
returns = returns.dropna()
returns.to_csv("data/daily_returns.csv")
print(returns.columns.tolist())

['oil', 'xle', 'xom', 'cvx', 'cop', 'bp']


In [258]:
import numpy as np
import scipy.stats as stats

def calculate_beta_single(stock_returns, factor_returns):
    X = np.column_stack([np.ones(len(factor_returns)), factor_returns])
    y = stock_returns.values
    coeffs, residuals, rank, s = np.linalg.lstsq(X, y, rcond=None)
    alpha, beta = coeffs
    y_pred = X @ coeffs
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (ss_res / ss_tot)
    resid_std = np.std(y - y_pred)
    return {"alpha": round(alpha, 4), "beta": round(beta, 3),
            "r_squared": round(r_squared, 3), "resid_std": round(resid_std, 4)}

In [259]:
estimation_window = returns.loc[:"2026-02-27"]
print(f"Estimation window: {len(estimation_window)} trading days")

baseline_oil_results = []
baseline_xle_results = []
for name in stock_names:
    r_oil = calculate_beta_single(estimation_window[name], estimation_window["oil"])
    r_oil["ticker"] = name
    baseline_oil_results.append(r_oil)
    
    r_xle = calculate_beta_single(estimation_window[name], estimation_window["xle"])
    r_xle["ticker"] = name
    baseline_xle_results.append(r_xle)

baseline_oil_df = pd.DataFrame(baseline_oil_results)[["ticker", "beta", "alpha", "r_squared", "resid_std"]]
baseline_xle_df = pd.DataFrame(baseline_xle_results)[["ticker", "beta", "alpha", "r_squared", "resid_std"]]

baseline_oil_df.to_csv("data/baseline_betas_oil.csv", index=False)
baseline_xle_df.to_csv("data/baseline_betas_xle.csv", index=False)

print("Beta vs OIL:")
display(baseline_oil_df)
print("Beta vs XLE:")
display(baseline_xle_df)

Estimation window: 123 trading days
Beta vs OIL:


,ticker,beta,alpha,r_squared,resid_std
0,xom,0.574,0.2355,0.508,1.0179
1,cvx,0.458,0.1267,0.394,1.0227
2,cop,0.699,0.1188,0.450,1.3902
3,bp,0.570,0.0963,0.376,1.3241


Beta vs XLE:


,ticker,beta,alpha,r_squared,resid_std
0,xom,1.045,0.0525,0.816,0.6220
1,cvx,0.905,-0.0333,0.748,0.6597
2,cop,1.336,-0.1166,0.800,0.8395
3,bp,1.016,-0.0813,0.579,1.0877


In [260]:
war_period = returns.loc["2026-02-20":"2026-06-30"].copy()

for name in stock_names:
    b_oil = baseline_oil_df[baseline_oil_df["ticker"] == name].iloc[0]
    predicted_oil = b_oil["alpha"] + b_oil["beta"] * war_period["oil"]
    war_period[f"{name}_abnormal_oil"] = war_period[name] - predicted_oil
    
    b_xle = baseline_xle_df[baseline_xle_df["ticker"] == name].iloc[0]
    predicted_xle = b_xle["alpha"] + b_xle["beta"] * war_period["xle"]
    war_period[f"{name}_abnormal_xle"] = war_period[name] - predicted_xle

war_period.to_csv("data/abnormal_returns_both.csv")

In [261]:
def build_oil_table(baseline_df, factor_col, abnormal_suffix):
    results = []
    for name in stock_names:
        baseline = baseline_df[baseline_df["ticker"] == name].iloc[0]
        car_total = war_days[f"{name}_abnormal_{abnormal_suffix}"].sum()
        
        if car_total < -5:
            interpretation = "Underperformed"
        elif car_total > 5:
            interpretation = "Outperformed"
        else:
            interpretation = "Tracked closely"
        
        results.append({
            "ticker": name,
            "beta_oil": baseline["beta"],
            "r_squared": baseline["r_squared"],
            "CAR_full_war_oil": round(car_total, 2),
            "interpretation": interpretation
        })
    return pd.DataFrame(results).sort_values("CAR_full_war_oil")

def build_xle_table(baseline_df, factor_col, abnormal_suffix):
    results = []
    for name in stock_names:
        baseline = baseline_df[baseline_df["ticker"] == name].iloc[0]
        car_total = war_days[f"{name}_abnormal_{abnormal_suffix}"].sum()
        war_result = calculate_beta_single(war_days[name], war_days[factor_col])
        
        if car_total < -5:
            interpretation = "Underperformed"
        elif car_total > 5:
            interpretation = "Outperformed"
        else:
            interpretation = "Tracked closely"
        
        results.append({
            "ticker": name,
            "baseline_beta_xle": baseline["beta"],
            "war_beta_xle": war_result["beta"],
            "r_squared": baseline["r_squared"],
            "CAR_full_war_xle": round(car_total, 2),
            "interpretation": interpretation
        })
    return pd.DataFrame(results).sort_values("CAR_full_war_xle")

car_oil_df = build_oil_table(baseline_oil_df, "oil", "oil")
car_xle_df = build_xle_table(baseline_xle_df, "xle", "xle")

car_oil_df.to_csv("data/car_full_war_oil.csv", index=False)
car_xle_df.to_csv("data/car_full_war_xle.csv", index=False)

print("CAR vs OIL model (beta = normal, pre-war sensitivity to oil):")
display(car_oil_df)
print("\nCAR vs XLE model (beta shown for both pre-war and during-war, to show if sector-relationship shifted):")
display(car_xle_df)

CAR vs OIL model (beta = normal, pre-war sensitivity to oil):


,ticker,beta_oil,r_squared,CAR_full_war_oil,interpretation
0,xom,0.574,0.508,-37.96,Underperformed
2,cop,0.699,0.450,-26.98,Underperformed
1,cvx,0.458,0.394,-26.02,Underperformed
3,bp,0.570,0.376,-18.33,Underperformed



CAR vs XLE model (beta shown for both pre-war and during-war, to show if sector-relationship shifted):


,ticker,baseline_beta_xle,war_beta_xle,r_squared,CAR_full_war_xle,interpretation
0,xom,1.045,1.120,0.816,-11.66,Underperformed
1,cvx,0.905,1.013,0.748,-3.73,Tracked closely
2,cop,1.336,1.184,0.800,6.22,Outperformed
3,bp,1.016,0.965,0.579,7.41,Outperformed
